In [41]:
%pip install -qU "google-cloud-aiplatform[agent_engines,adk]>=1.112"

In [42]:
import os

import vertexai

from google.colab import auth


auth.authenticate_user()

PROJECT_ID = "qwiklabs-gcp-03-918ac2e61dcf"
DEFAULT_LOCATION = "us-central1"
CONTACT_EMAIL = "david.bullard364@gmail.com"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = DEFAULT_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["WEATHER_CONTACT"] = CONTACT_EMAIL

vertexai.init(
    project=PROJECT_ID,
    location=DEFAULT_LOCATION,
)

print("Authentication successful.")
print("Project:", PROJECT_ID)
print("Default location:", DEFAULT_LOCATION)

Authentication successful.
Project: qwiklabs-gcp-03-918ac2e61dcf
Default location: us-central1


In [43]:
!gcloud config set project {PROJECT_ID}
!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

import google.adk

print(
    "Google ADK version:",
    getattr(google.adk, "__version__", "version unavailable"),
)

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Google ADK version: 2.13.0


In [44]:
import json
import time

from functools import lru_cache
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


USER_AGENT = (
    f"ADKMultiAgentWeather/1.0 ({CONTACT_EMAIL})"
)

REQUEST_TIMEOUT = 20
_last_geocode_request = 0.0

In [45]:
def _fetch_json(
    url: str,
    accept: str = "application/geo+json",
) -> dict:
    """Retrieve and decode a JSON document.

    Args:
        url: Fully qualified URL to request.
        accept: HTTP Accept header value.

    Returns:
        The decoded JSON response.

    Raises:
        HTTPError: If the server returns an HTTP error.
        URLError: If the server cannot be reached.
        TimeoutError: If the request times out.
    """
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": accept,
        },
    )

    with urlopen(
        request,
        timeout=REQUEST_TIMEOUT,
    ) as response:
        return json.load(response)


@lru_cache(maxsize=256)
def _get_coordinates(
    city: str,
    state: str,
) -> tuple[float, float]:
    """Convert a U.S. city and state to coordinates.

    Args:
        city: U.S. city name.
        state: State name or two-letter abbreviation.

    Returns:
        A latitude-and-longitude tuple.

    Raises:
        ValueError: If no matching U.S. location is found.
    """
    global _last_geocode_request

    city = city.strip()
    state = state.strip()

    if not city or not state:
        raise ValueError(
            "Both city and state are required."
        )

    elapsed = (
        time.monotonic() - _last_geocode_request
    )

    time.sleep(
        max(0.0, 1.0 - elapsed)
    )

    parameters = urlencode(
        {
            "city": city,
            "state": state,
            "countrycodes": "us",
            "format": "jsonv2",
            "limit": 1,
        }
    )

    url = (
        "https://nominatim.openstreetmap.org/search?"
        f"{parameters}"
    )

    try:
        locations = _fetch_json(
            url,
            accept="application/json",
        )
    finally:
        _last_geocode_request = time.monotonic()

    if not locations:
        raise ValueError(
            f"No U.S. location was found for "
            f"{city}, {state}."
        )

    return (
        float(locations[0]["lat"]),
        float(locations[0]["lon"]),
    )


print("Geocoding functions created.")

Geocoding functions created.


In [46]:
def _measurement_value(
    properties: dict,
    field: str,
):
    """Extract a measurement from an NWS observation.

    Args:
        properties: NWS observation properties.
        field: Name of the measurement field.

    Returns:
        The measurement value or None.
    """
    measurement = properties.get(field) or {}
    return measurement.get("value")


def _condition_advisory(
    temperature_f: float | None,
    wind_mph: float | None,
    official_alerts: list[dict],
) -> str:
    """Create an advisory from alerts and conditions.

    Args:
        temperature_f: Temperature in Fahrenheit.
        wind_mph: Wind speed in miles per hour.
        official_alerts: Active NWS alerts.

    Returns:
        A concise weather advisory.
    """
    if official_alerts:
        alert_names = [
            alert.get("event")
            for alert in official_alerts
            if alert.get("event")
        ]

        if alert_names:
            return (
                "Official NWS alert: "
                + "; ".join(alert_names)
            )

    advisories = []

    if temperature_f is not None:
        if temperature_f >= 95:
            advisories.append(
                "high heat conditions"
            )
        elif temperature_f <= 32:
            advisories.append(
                "freezing conditions"
            )

    if (
        wind_mph is not None
        and wind_mph >= 30
    ):
        advisories.append("strong winds")

    if advisories:
        return (
            "Use caution due to "
            + "; ".join(advisories)
            + "."
        )

    return (
        "No active NWS alert or threshold-based "
        "weather advisory."
    )


print("Weather helper functions created.")

Weather helper functions created.


In [47]:
def get_current_weather(
    city: str,
    state: str,
) -> dict:
    """Get current and forecast weather for a U.S. location.

    The tool retrieves current observations, the next forecast
    period, active alerts, and a condition-based advisory.

    Args:
        city: U.S. city name, such as Austin.
        state: State name or abbreviation, such as Texas or TX.

    Returns:
        A dictionary containing the observation, forecast,
        alerts, advisory, and source information.
    """
    city = city.strip()
    state = state.strip()

    if not city or not state:
        return {
            "status": "error",
            "message": (
                "Both city and state are required."
            ),
        }

    try:
        latitude, longitude = _get_coordinates(
            city,
            state,
        )

        point_url = (
            "https://api.weather.gov/points/"
            f"{latitude:.4f},{longitude:.4f}"
        )

        point_document = _fetch_json(point_url)
        point_properties = (
            point_document["properties"]
        )

        stations_document = _fetch_json(
            point_properties[
                "observationStations"
            ]
        )

        station_features = (
            stations_document.get(
                "features",
                [],
            )
        )

        if not station_features:
            raise ValueError(
                "No NWS observation station was found."
            )

        observation_properties = None
        selected_station = None

        for station in station_features[:5]:
            station_url = station.get("id")

            if not station_url:
                continue

            try:
                observation = _fetch_json(
                    f"{station_url}/observations/latest"
                )

                candidate = observation["properties"]

                candidate_temperature = (
                    _measurement_value(
                        candidate,
                        "temperature",
                    )
                )

                if candidate_temperature is not None:
                    observation_properties = candidate
                    selected_station = (
                        station_url.rsplit("/", 1)[-1]
                    )
                    break

            except (
                HTTPError,
                URLError,
                TimeoutError,
                KeyError,
                TypeError,
            ):
                continue

        if observation_properties is None:
            raise ValueError(
                "Nearby NWS stations did not report "
                "a usable current temperature."
            )

        temperature_c = _measurement_value(
            observation_properties,
            "temperature",
        )

        wind_mps = _measurement_value(
            observation_properties,
            "windSpeed",
        )

        humidity = _measurement_value(
            observation_properties,
            "relativeHumidity",
        )

        temperature_f = round(
            (temperature_c * 9 / 5) + 32,
            1,
        )

        wind_mph = (
            round(wind_mps * 2.23694, 1)
            if wind_mps is not None
            else None
        )

        forecast_document = _fetch_json(
            point_properties["forecast"]
        )

        forecast_periods = (
            forecast_document["properties"][
                "periods"
            ]
        )

        if not forecast_periods:
            raise ValueError(
                "No NWS forecast periods were returned."
            )

        next_period = forecast_periods[0]

        alerts_url = (
            "https://api.weather.gov/alerts/active?"
            + urlencode(
                {
                    "point": (
                        f"{latitude:.4f},"
                        f"{longitude:.4f}"
                    )
                }
            )
        )

        alerts_document = _fetch_json(
            alerts_url
        )

        official_alerts = []

        for feature in alerts_document.get(
            "features",
            [],
        )[:5]:
            alert = feature.get(
                "properties",
                {},
            )

            official_alerts.append(
                {
                    "event": alert.get("event"),
                    "severity": alert.get(
                        "severity"
                    ),
                    "headline": alert.get(
                        "headline"
                    ),
                    "expires": alert.get(
                        "expires"
                    ),
                }
            )

        advisory = _condition_advisory(
            temperature_f=temperature_f,
            wind_mph=wind_mph,
            official_alerts=official_alerts,
        )

        return {
            "status": "success",
            "location": f"{city}, {state}",
            "coordinates": {
                "latitude": latitude,
                "longitude": longitude,
            },
            "current": {
                "station": selected_station,
                "observed_at": (
                    observation_properties.get(
                        "timestamp"
                    )
                ),
                "description": (
                    observation_properties.get(
                        "textDescription"
                    )
                    or "No description reported"
                ),
                "temperature_f": temperature_f,
                "relative_humidity_percent": (
                    round(humidity, 1)
                    if humidity is not None
                    else None
                ),
                "wind_mph": wind_mph,
                "wind_direction_degrees": (
                    _measurement_value(
                        observation_properties,
                        "windDirection",
                    )
                ),
            },
            "next_forecast": {
                "name": next_period.get("name"),
                "temperature": next_period.get(
                    "temperature"
                ),
                "temperature_unit": (
                    next_period.get(
                        "temperatureUnit"
                    )
                ),
                "wind": (
                    f"{next_period.get('windSpeed')} "
                    f"{next_period.get('windDirection')}"
                ),
                "summary": next_period.get(
                    "shortForecast"
                ),
                "details": next_period.get(
                    "detailedForecast"
                ),
            },
            "official_alerts": official_alerts,
            "advisory": advisory,
            "sources": [
                "National Weather Service",
                "OpenStreetMap Nominatim",
            ],
        }

    except HTTPError as error:
        return {
            "status": "error",
            "location": f"{city}, {state}",
            "message": (
                "Weather service returned HTTP "
                f"{error.code}: {error.reason}"
            ),
        }

    except (
        URLError,
        TimeoutError,
    ) as error:
        return {
            "status": "error",
            "location": f"{city}, {state}",
            "message": (
                "Could not reach a weather service: "
                f"{error}"
            ),
        }

    except (
        KeyError,
        IndexError,
        TypeError,
        ValueError,
    ) as error:
        return {
            "status": "error",
            "location": f"{city}, {state}",
            "message": str(error),
        }


print("NWS weather tool created.")

NWS weather tool created.


In [48]:
from google.adk.tools import ToolContext


In [49]:
weather_tool_test = get_current_weather(
    "Austin",
    "Texas",
)

assert weather_tool_test["status"] == "success", (
    weather_tool_test
)

assert (
    weather_tool_test["current"][
        "temperature_f"
    ]
    is not None
)

assert weather_tool_test["next_forecast"]["name"]

print("Weather tool test passed.")
print(
    json.dumps(
        weather_tool_test,
        indent=2,
    )
)

Weather tool test passed.
{
  "status": "success",
  "location": "Austin, Texas",
  "coordinates": {
    "latitude": 30.2649261,
    "longitude": -97.7474086
  },
  "current": {
    "station": "KAUS",
    "observed_at": "2026-10-26T19:00:00+00:00",
    "description": "Mostly Clear",
    "temperature_f": 78.8,
    "relative_humidity_percent": 47.6,
    "wind_mph": null,
    "wind_direction_degrees": null
  },
  "next_forecast": {
    "name": "This Afternoon",
    "temperature": 84,
    "temperature_unit": "F",
    "wind": "5 mph SE",
    "summary": "Partly Sunny",
    "details": "Partly sunny, with a high near 84. Southeast wind around 5 mph."
  },
  "official_alerts": [],
  "advisory": "No active NWS alert or threshold-based weather advisory.",
  "sources": [
    "National Weather Service",
    "OpenStreetMap Nominatim"
  ]
}


In [50]:
from google import genai


# The built-in ADK Google Search tool requires a Gemini 2 model.
CANDIDATE_MODELS = [
    "gemini-2.5-flash",
    "gemini-2.0-flash",
]

CANDIDATE_LOCATIONS = [
    "us-central1",
    "global",
]


def find_available_gemini_2_model(
) -> tuple[str, str]:
    """Find an available Gemini 2 model and location.

    Returns:
        A tuple containing the model ID and location.

    Raises:
        RuntimeError: If no candidate model is available.
    """
    failures = []

    for candidate_location in CANDIDATE_LOCATIONS:
        client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location=candidate_location,
        )

        for candidate_model in CANDIDATE_MODELS:
            try:
                response = (
                    client.models.generate_content(
                        model=candidate_model,
                        contents=(
                            "Respond with only the word OK."
                        ),
                    )
                )

                print(
                    "AVAILABLE:",
                    candidate_model,
                    "in",
                    candidate_location,
                )

                print(
                    "Test response:",
                    response.text,
                )

                return (
                    candidate_model,
                    candidate_location,
                )

            except Exception as error:
                error_summary = str(error).split(
                    "\n"
                )[0]

                failures.append(
                    {
                        "model": candidate_model,
                        "location": (
                            candidate_location
                        ),
                        "error": error_summary,
                    }
                )

                print(
                    "Unavailable:",
                    candidate_model,
                    "in",
                    candidate_location,
                )

    raise RuntimeError(
        "No compatible Gemini 2 model was "
        f"available. Failures: {failures}"
    )


MODEL_ID, MODEL_LOCATION = (
    find_available_gemini_2_model()
)

os.environ[
    "GOOGLE_CLOUD_LOCATION"
] = MODEL_LOCATION

vertexai.init(
    project=PROJECT_ID,
    location=MODEL_LOCATION,
)

print("\nSelected model:", MODEL_ID)
print("Selected location:", MODEL_LOCATION)

AVAILABLE: gemini-2.5-flash in us-central1
Test response: OK

Selected model: gemini-2.5-flash
Selected location: us-central1


In [51]:
# Cell 10 — Instructions for the three-agent system

WEATHER_AGENT_INSTRUCTIONS = """
You are the specialized weather agent.

Your responsibilities:

1. Handle U.S. weather requests.
2. Require both a city and state.
3. Always call get_weather_and_remember_location.
4. Never invent observations, forecasts, or alerts.
5. Do not use Google Search.
6. If the tool returns an error, explain it clearly.
7. The weather tool automatically saves the location and
   transfers control to search_agent after a successful result.
8. Do not attempt a second transfer after the tool succeeds.
"""


SEARCH_AGENT_INSTRUCTIONS = """
You are the specialized local search and recommendations agent.

The current session location is:

City: {selected_city?}
State: {selected_state?}

Your responsibilities:

1. If you have just received a successful weather-tool result:
   a. Briefly summarize the current weather and forecast from
      that tool result.
   b. Do not perform a Google Search yet.
   c. Ask exactly one follow-up question:
      "What are you interested in doing in or near
      {selected_city?}, {selected_state?}?"
   d. Give brief examples such as restaurants, family activities,
      outdoor recreation, museums, shopping, nightlife, or events.
   e. Stop and wait for the user's response.

2. When the user subsequently identifies an interest:
   a. Use google_search.
   b. Search for current recommendations in or near
      {selected_city?}, {selected_state?}.
   c. Provide approximately five useful recommendations.
   d. For each recommendation, include:
      - Name
      - Why it matches the user's interest
      - Location or neighborhood when available
      - Useful current details when available
      - Source attribution

3. Do not invent businesses, attractions, hours, prices,
   ratings, events, or availability.

4. If no selected city or state is available, transfer control
   to root_agent so it can request the missing location.

5. If the user requests weather for another location, transfer
   control to weather_agent.

6. If the user changes only their activity preference, continue
   using the current saved city and state.
"""


ROOT_AGENT_INSTRUCTIONS = """
You are the coordinating root agent for a multi-agent local
information system.

The current saved location, if available, is:

City: {selected_city?}
State: {selected_state?}

Delegation rules:

1. Transfer weather questions to weather_agent.
2. Transfer requests for local activities, businesses,
   restaurants, attractions, events, recommendations, current
   facts, recent developments, or web research to search_agent.
3. If the user provides an activity preference after receiving
   weather, transfer that request to search_agent.
4. Do not retrieve or invent weather information yourself.
5. Do not claim to have searched Google yourself.
6. Do not call specialist tools yourself.
7. If a weather request lacks either a city or state, ask the
   user to provide the missing information.
8. Preserve the saved location until the user selects a new one.
9. When the user selects a new city and state, transfer the
   request to weather_agent so session state is updated.
"""

In [52]:
# Cell 10 — Instructions for the three-agent system

WEATHER_AGENT_INSTRUCTIONS = """
You are the specialized weather agent.

Your responsibilities:

1. Handle U.S. weather requests.
2. Require both a city and state.
3. Always call get_weather_and_remember_location.
4. Never invent observations, forecasts, or alerts.
5. Do not use Google Search.
6. If the tool returns an error, explain it clearly.
7. The weather tool automatically saves the location and
   transfers control to search_agent after a successful result.
8. Do not attempt a second transfer after the tool succeeds.
"""


SEARCH_AGENT_INSTRUCTIONS = """
You are the specialized local search and recommendations agent.

The current session location is:

City: {selected_city?}
State: {selected_state?}

Your responsibilities:

1. If you have just received a successful weather-tool result:
   a. Briefly summarize the current weather and forecast from
      that tool result.
   b. Do not perform a Google Search yet.
   c. Ask exactly one follow-up question:
      "What are you interested in doing in or near
      {selected_city?}, {selected_state?}?"
   d. Give brief examples such as restaurants, family activities,
      outdoor recreation, museums, shopping, nightlife, or events.
   e. Stop and wait for the user's response.

2. When the user subsequently identifies an interest:
   a. Use google_search.
   b. Search for current recommendations in or near
      {selected_city?}, {selected_state?}.
   c. Provide approximately five useful recommendations.
   d. For each recommendation, include:
      - Name
      - Why it matches the user's interest
      - Location or neighborhood when available
      - Useful current details when available
      - Source attribution

3. Do not invent businesses, attractions, hours, prices,
   ratings, events, or availability.

4. If no selected city or state is available, transfer control
   to root_agent so it can request the missing location.

5. If the user requests weather for another location, transfer
   control to weather_agent.

6. If the user changes only their activity preference, continue
   using the current saved city and state.
"""


ROOT_AGENT_INSTRUCTIONS = """
You are the coordinating root agent for a multi-agent local
information system.

The current saved location, if available, is:

City: {selected_city?}
State: {selected_state?}

Delegation rules:

1. Transfer weather questions to weather_agent.
2. Transfer requests for local activities, businesses,
   restaurants, attractions, events, recommendations, current
   facts, recent developments, or web research to search_agent.
3. If the user provides an activity preference after receiving
   weather, transfer that request to search_agent.
4. Do not retrieve or invent weather information yourself.
5. Do not claim to have searched Google yourself.
6. Do not call specialist tools yourself.
7. If a weather request lacks either a city or state, ask the
   user to provide the missing information.
8. Preserve the saved location until the user selects a new one.
9. When the user selects a new city and state, transfer the
   request to weather_agent so session state is updated.
"""

In [53]:
# Cell 12 — Verify agent hierarchy and assigned tools

def registered_tool_name(tool) -> str:
    """Return a readable name for a registered ADK tool."""
    name = getattr(tool, "name", None)

    if name:
        return name

    function_name = getattr(
        tool,
        "__name__",
        None,
    )

    if function_name:
        return function_name

    return type(tool).__name__


registered_sub_agents = {
    agent.name
    for agent in root_agent.sub_agents
}

weather_tool_names = {
    registered_tool_name(tool)
    for tool in weather_agent.tools
}

search_tool_names = {
    registered_tool_name(tool)
    for tool in search_agent.tools
}

assert registered_sub_agents == {
    "weather_agent",
    "search_agent",
}, registered_sub_agents

assert (
    "get_weather_and_remember_location"
    in weather_tool_names
), weather_tool_names

assert any(
    "google_search" in name.lower()
    or "googlesearch" in name.lower()
    for name in search_tool_names
), search_tool_names

print("Agent hierarchy verified.")
print(
    "Registered subagents:",
    sorted(registered_sub_agents),
)

print(
    "Weather-agent tools:",
    sorted(weather_tool_names),
)

print(
    "Search-agent tools:",
    sorted(search_tool_names),
)

Agent hierarchy verified.
Registered subagents: ['search_agent', 'weather_agent']
Weather-agent tools: ['get_weather_and_remember_location']
Search-agent tools: ['google_search']


In [54]:
# Cell 13 — Create application and event-inspection helpers

import uuid

from IPython.display import HTML, Markdown, display
from vertexai.agent_engines import AdkApp


app = AdkApp(
    agent=root_agent,
)


def _as_dictionary(value) -> dict:
    """Convert an ADK object to a dictionary.

    Args:
        value: Dictionary or Pydantic-compatible ADK object.

    Returns:
        A normalized dictionary.
    """
    if isinstance(value, dict):
        return value

    if hasattr(value, "model_dump"):
        return value.model_dump(
            mode="json",
            exclude_none=True,
        )

    raise TypeError(
        "Unsupported ADK event type: "
        f"{type(value).__name__}"
    )


def _event_parts(
    event_dictionary: dict,
) -> list[dict]:
    """Extract normalized content parts from an ADK event."""
    content = (
        event_dictionary.get("content")
        or {}
    )

    if not isinstance(content, dict):
        content = _as_dictionary(content)

    parts = content.get("parts") or []

    return [
        (
            part
            if isinstance(part, dict)
            else _as_dictionary(part)
        )
        for part in parts
    ]


def _get_dictionary_value(
    dictionary: dict,
    snake_case_key: str,
    camel_case_key: str,
):
    """Read a snake_case or camelCase dictionary value."""
    if not isinstance(dictionary, dict):
        return None

    value = dictionary.get(snake_case_key)

    if value is not None:
        return value

    return dictionary.get(camel_case_key)


def _event_grounding_metadata(
    event_dictionary: dict,
) -> dict | None:
    """Return Google Search grounding metadata from an event."""
    metadata = _get_dictionary_value(
        event_dictionary,
        "grounding_metadata",
        "groundingMetadata",
    )

    if isinstance(metadata, dict):
        return metadata

    return None


def print_event_summary(
    event_number: int,
    event_dictionary: dict,
    is_final_response: bool,
) -> None:
    """Print routing, tool, state, and grounding details."""
    author = event_dictionary.get(
        "author",
        "unknown",
    )

    branch = event_dictionary.get(
        "branch",
        "",
    )

    print(
        f"\nEVENT {event_number}"
        f" | author={author}"
        f" | branch={branch or 'default'}"
        f" | final={is_final_response}"
    )

    actions = (
        event_dictionary.get("actions")
        or {}
    )

    transfer = _get_dictionary_value(
        actions,
        "transfer_to_agent",
        "transferToAgent",
    )

    if transfer:
        print(
            "  TRANSFER:",
            transfer,
        )

    state_delta = _get_dictionary_value(
        actions,
        "state_delta",
        "stateDelta",
    )

    if state_delta:
        print(
            "  STATE UPDATE:",
            state_delta,
        )

    for part in _event_parts(
        event_dictionary
    ):
        function_call = _get_dictionary_value(
            part,
            "function_call",
            "functionCall",
        )

        if function_call:
            print(
                "  TOOL CALL:",
                function_call.get("name"),
            )
            print(
                "  ARGUMENTS:",
                function_call.get("args", {}),
            )

        function_response = _get_dictionary_value(
            part,
            "function_response",
            "functionResponse",
        )

        if function_response:
            print(
                "  TOOL RESPONSE:",
                function_response.get("name"),
            )

        text = part.get("text")

        if text:
            preview = text.replace(
                "\n",
                " ",
            )[:500]

            print(
                "  TEXT:",
                preview,
            )

    grounding_metadata = (
        _event_grounding_metadata(
            event_dictionary
        )
    )

    if grounding_metadata:
        chunks = _get_dictionary_value(
            grounding_metadata,
            "grounding_chunks",
            "groundingChunks",
        ) or []

        print(
            "  GOOGLE SEARCH GROUNDING:",
            len(chunks),
            "source chunk(s)",
        )

        search_entry_point = (
            _get_dictionary_value(
                grounding_metadata,
                "search_entry_point",
                "searchEntryPoint",
            )
            or {}
        )

        rendered_content = (
            _get_dictionary_value(
                search_entry_point,
                "rendered_content",
                "renderedContent",
            )
        )

        if rendered_content:
            display(
                HTML(rendered_content)
            )


print(
    "Persistent multi-agent application and "
    "event helpers created."
)

Persistent multi-agent application and event helpers created.


In [55]:
# Cell 14 — Persistent multi-turn conversation runner

CONVERSATION_USER_ID = "local-planner-user"
conversation_session_id = None


async def start_new_conversation() -> str:
    """Create a new persistent multi-agent conversation.

    Returns:
        The new session identifier.
    """
    global conversation_session_id

    conversation_session_id = str(
        uuid.uuid4()
    )

    await app.async_create_session(
        user_id=CONVERSATION_USER_ID,
        session_id=conversation_session_id,
    )

    print(
        "New conversation created:",
        conversation_session_id,
    )

    return conversation_session_id


async def send_conversation_message(
    prompt: str,
    show_events: bool = True,
) -> dict:
    """Send a message through the persistent conversation.

    Args:
        prompt: User message sent to the active agent.
        show_events: Whether to print ADK events.

    Returns:
        The final answer, transcript, agents, transfers, tools,
        state updates, grounding metadata, and events.
    """
    global conversation_session_id

    prompt = prompt.strip()

    if not prompt:
        raise ValueError(
            "The conversation prompt cannot be empty."
        )

    if conversation_session_id is None:
        await start_new_conversation()

    events = []
    authors = []
    tool_calls = []
    transfers = []
    state_updates = []
    grounding_metadata_items = []
    all_text_fragments = []
    final_text_fragments = []

    print("\n" + "=" * 72)
    print("USER")
    print(prompt)
    print("=" * 72)

    event_number = 0

    async for event in app.async_stream_query(
        user_id=CONVERSATION_USER_ID,
        session_id=conversation_session_id,
        message=prompt,
    ):
        event_number += 1

        is_final_response = False

        if hasattr(event, "is_final_response"):
            try:
                is_final_response = (
                    event.is_final_response()
                )
            except TypeError:
                is_final_response = False

        event_dictionary = _as_dictionary(
            event
        )

        events.append(event_dictionary)

        author = event_dictionary.get(
            "author"
        )

        if author:
            authors.append(author)

        actions = (
            event_dictionary.get("actions")
            or {}
        )

        transfer = _get_dictionary_value(
            actions,
            "transfer_to_agent",
            "transferToAgent",
        )

        if transfer:
            transfers.append(transfer)

        state_delta = _get_dictionary_value(
            actions,
            "state_delta",
            "stateDelta",
        )

        if state_delta:
            state_updates.append(state_delta)

        grounding_metadata = (
            _event_grounding_metadata(
                event_dictionary
            )
        )

        if grounding_metadata:
            grounding_metadata_items.append(
                grounding_metadata
            )

        event_text_fragments = []

        for part in _event_parts(
            event_dictionary
        ):
            text = part.get("text")

            if text:
                event_text_fragments.append(text)
                all_text_fragments.append(text)

            function_call = _get_dictionary_value(
                part,
                "function_call",
                "functionCall",
            )

            if function_call:
                tool_name = function_call.get(
                    "name"
                )

                if tool_name:
                    tool_calls.append(tool_name)

                if (
                    tool_name
                    == "transfer_to_agent"
                ):
                    arguments = (
                        function_call.get("args")
                        or {}
                    )

                    destination = (
                        arguments.get("agent_name")
                        or arguments.get(
                            "agentName"
                        )
                    )

                    if destination:
                        transfers.append(
                            destination
                        )

        if (
            is_final_response
            and event_text_fragments
        ):
            final_text_fragments.extend(
                event_text_fragments
            )

        if show_events:
            print_event_summary(
                event_number=event_number,
                event_dictionary=event_dictionary,
                is_final_response=is_final_response,
            )

    transcript = "\n".join(
        all_text_fragments
    ).strip()

    final_answer = "\n".join(
        final_text_fragments
    ).strip()

    if not final_answer:
        final_answer = transcript

    # Preserve order while removing duplicates.
    unique_authors = list(
        dict.fromkeys(authors)
    )

    unique_tool_calls = list(
        dict.fromkeys(tool_calls)
    )

    unique_transfers = list(
        dict.fromkeys(transfers)
    )

    print("\n" + "=" * 72)
    print("FINAL AGENT RESPONSE")
    print("=" * 72)

    if final_answer:
        display(
            Markdown(final_answer)
        )
    else:
        print(
            "The agents returned no textual response."
        )

    print(
        "\nAgents used:",
        unique_authors,
    )

    print(
        "Transfers:",
        unique_transfers,
    )

    print(
        "Tool calls:",
        unique_tool_calls,
    )

    print(
        "Grounded search events:",
        len(grounding_metadata_items),
    )

    return {
        "prompt": prompt,
        "answer": final_answer,
        "transcript": transcript,
        "authors": unique_authors,
        "tool_calls": unique_tool_calls,
        "transfers": unique_transfers,
        "state_updates": state_updates,
        "grounding_metadata": (
            grounding_metadata_items
        ),
        "events": events,
        "session_id": conversation_session_id,
    }


print(
    "Persistent conversation runner created."
)

Persistent conversation runner created.


In [56]:
# Cell 15 — Test weather retrieval and deterministic handoff

await start_new_conversation()

weather_handoff_result = (
    await send_conversation_message(
        (
            "What is the current weather in "
            "Leesburg, Virginia? Include the next "
            "forecast period and any active alerts."
        ),
        show_events=True,
    )
)

assert "weather_agent" in (
    weather_handoff_result["authors"]
), (
    "The root agent did not delegate to weather_agent."
)

assert (
    "get_weather_and_remember_location"
    in weather_handoff_result["tool_calls"]
), (
    "The weather agent did not call its weather tool."
)

assert "search_agent" in (
    weather_handoff_result["authors"]
), (
    "Control did not transfer to search_agent."
)

assert "search_agent" in (
    weather_handoff_result["transfers"]
), (
    "No transfer-to-search event was recorded."
)

combined_state_updates = {}

for state_update in (
    weather_handoff_result["state_updates"]
):
    combined_state_updates.update(
        state_update
    )

assert (
    combined_state_updates.get(
        "selected_city"
    )
    == "Leesburg"
), combined_state_updates

assert (
    combined_state_updates.get(
        "selected_state"
    )
    == "Virginia"
), combined_state_updates

answer_lower = (
    weather_handoff_result["answer"].lower()
)

assert any(
    phrase in answer_lower
    for phrase in (
        "interested in",
        "what would you like",
        "what are you interested",
    )
), (
    "The search agent did not ask about "
    "the user's interests."
)

print(
    "\nPASS: Root agent delegated to weather_agent."
)

print(
    "PASS: Weather tool stored the selected location."
)

print(
    "PASS: Weather tool transferred to search_agent."
)

print(
    "PASS: Search agent asked about local interests."
)

New conversation created: 59cdf4cb-a2aa-48d4-b8bd-847d0b3bab5d

USER
What is the current weather in Leesburg, Virginia? Include the next forecast period and any active alerts.

EVENT 1 | author=root_agent | branch=default | final=False
  TOOL CALL: transfer_to_agent
  ARGUMENTS: {'agent_name': 'weather_agent'}

EVENT 2 | author=root_agent | branch=default | final=False
  TRANSFER: weather_agent
  TOOL RESPONSE: transfer_to_agent

EVENT 3 | author=weather_agent | branch=default | final=False
  TOOL CALL: get_weather_and_remember_location
  ARGUMENTS: {'state': 'Virginia', 'city': 'Leesburg'}
[SESSION STATE] Saved location: Leesburg, Virginia

EVENT 4 | author=weather_agent | branch=default | final=False
  STATE UPDATE: {'selected_city': 'Leesburg', 'selected_state': 'Virginia', 'recommendation_stage': 'awaiting_user_interests'}
  TOOL RESPONSE: get_weather_and_remember_location

EVENT 5 | author=weather_agent | branch=default | final=False
  TEXT: The current weather in Leesburg, Virgin

The current weather in Leesburg, Virginia is clear with a temperature of 71.6°F. The humidity is 37.9%, and the wind is blowing from the west-northwest at 24.9 mph.

For "This Afternoon", the forecast is sunny, with a high near 70°F and a northwest wind around 6 mph.

There are no active National Weather Service alerts or advisories for Leesburg, Virginia.

What are you interested in doing in or near Leesburg, Virginia? For example, are you looking for restaurants, family activities, outdoor recreation, museums, shopping, nightlife, or local events?


Agents used: ['root_agent', 'weather_agent', 'search_agent']
Transfers: ['weather_agent', 'search_agent']
Tool calls: ['transfer_to_agent', 'get_weather_and_remember_location']
Grounded search events: 0

PASS: Root agent delegated to weather_agent.
PASS: Weather tool stored the selected location.
PASS: Weather tool transferred to search_agent.
PASS: Search agent asked about local interests.


In [57]:
# Cell 16 — Test recommendation search in the same session

recommendation_result = (
    await send_conversation_message(
        (
            "I am interested in family-friendly outdoor "
            "activities appropriate for two young children. "
            "Please find current recommendations."
        ),
        show_events=True,
    )
)

assert "search_agent" in (
    recommendation_result["authors"]
), (
    "The follow-up was not handled by search_agent."
)

assert recommendation_result["answer"], (
    "The search agent returned no recommendations."
)

assert (
    recommendation_result[
        "grounding_metadata"
    ]
), (
    "No Google Search grounding metadata was returned. "
    "Verify that the search agent used google_search and "
    "that the selected Gemini model supports grounding."
)

print(
    "\nPASS: Search agent handled the user's "
    "activity preference."
)

print(
    "PASS: Google Search grounding metadata "
    "was returned."
)

print(
    "PASS: Recommendations used the location "
    "saved in the existing session."
)


USER
I am interested in family-friendly outdoor activities appropriate for two young children. Please find current recommendations.

EVENT 1 | author=search_agent | branch=default | final=False
  TEXT: Here are some family-friendly outdoor activity recommendations in or near Leesburg, Virginia, suitable for young children:    1. **Temple Hall Farm Regional Park** - Leesburg, VA     Why it matches: This working farm allows visitors to meet a variety of farm animals and experience daily farm life. It features a play yard, nature trail, hill slides, and jumping pillows. They also host seasonal events like pumpkin patches and sunflower fields.     Useful current details: General visits are free fr
  GOOGLE SEARCH GROUNDING: 18 source chunk(s)



FINAL AGENT RESPONSE


Here are some family-friendly outdoor activity recommendations in or near Leesburg, Virginia, suitable for young children: 

 1. **Temple Hall Farm Regional Park** - Leesburg, VA
    Why it matches: This working farm allows visitors to meet a variety of farm animals and experience daily farm life. It features a play yard, nature trail, hill slides, and jumping pillows. They also host seasonal events like pumpkin patches and sunflower fields.
    Useful current details: General visits are free from April through September (closed Mondays). October typically requires a separate seasonal ticket for events like the Pumpkin Patch & Flower Fields, which includes wagon rides, animal visits, inflatable pillows, and hill slides. Dogs and other pets are not permitted.

 2. **Leesburg Animal Park** - Leesburg, VA
    Why it matches: This family-owned zoo offers hands-on animal encounters with both domestic and exotic animals. Children can pet and feed farm animals like goats, sheep, and llamas in a contact area. The park also has pony rides, wagon rides, and an outdoor play area.
    Useful current details: The park features exotic animals such as zebras, lemurs, and sloths. There's also an indoor play zone, making it a good option for various weather conditions. Seasonal festivals like "Pumpkin Village" are held here.

 3. **Ida Lee Park** - Leesburg, VA
    Why it matches: This large park is considered the "crown jewel" of Leesburg's park system and offers two playgrounds, picnic areas, and walking trails. It provides ample space for children to run and play.
    Useful current details: The park also includes the Rust Library playground, which is surrounded by trees for shade. The AV Symington Aquatics Center, an outdoor waterpark with slides, a lazy river, and a splash area, is also located within Ida Lee Park, perfect for summer fun.

 4. **Douglass High School Park and Playground** - Leesburg, VA
    Why it matches: This park features an incredible playground with unique climbing structures that celebrate African American history and culture, making it an engaging and fun place for kids of all ages.
    Useful current details: The playground includes structures shaped like the National Museum of African American History and Culture and the Selma Bridge. It also has ball fields, a skating rink, and park benches near a creek.

 5. **Various Playgrounds in Leesburg**
    Why it matches: Leesburg has several well-maintained parks with playgrounds that are perfect for young children to burn off energy.
    Useful current details:
    *   **Rotary Park** (0.1 mi NW of Leesburg) offers a variety of play structures, swings, slides, and climbing walls, with ample seating and a picnic area.
    *   **Brandon Park** (0.1 mi W of Leesburg) features play structures, swings, and slides, along with green spaces and walking trails.
    *   **Robinson Park** (1 mi N of Leesburg) provides play structures, swings, and slides, as well as picnic areas.
    *   **Foxridge Park** (2 mi NW of Leesburg) has play structures, swings, and climbing equipment, along with picnic areas, walking trails, and green space.


Agents used: ['search_agent']
Transfers: []
Tool calls: []
Grounded search events: 1

PASS: Search agent handled the user's activity preference.
PASS: Google Search grounding metadata was returned.
PASS: Recommendations used the location saved in the existing session.


In [58]:
# Cell 17 — Display multi-agent grading evidence

print("=" * 72)
print("MULTI-AGENT CONVERSATION SUMMARY")
print("=" * 72)

all_demonstrated_agents = set(
    weather_handoff_result["authors"]
    + recommendation_result["authors"]
)

assert "root_agent" in all_demonstrated_agents
assert "weather_agent" in all_demonstrated_agents
assert "search_agent" in all_demonstrated_agents

print(
    "Persistent session:",
    conversation_session_id,
)

print("\nAgents demonstrated:")

for agent_name in sorted(
    all_demonstrated_agents
):
    print(" -", agent_name)

print(
    "\nWeather-turn transfers:",
    weather_handoff_result["transfers"],
)

print(
    "Weather-turn tool calls:",
    weather_handoff_result["tool_calls"],
)

print(
    "Recommendation-turn tool calls:",
    recommendation_result["tool_calls"],
)

print(
    "Google Search grounding events:",
    len(
        recommendation_result[
            "grounding_metadata"
        ]
    ),
)

print(
    "\nPASS: Root, weather, and search agents "
    "were all demonstrated."
)

print(
    "PASS: Conversation state persisted across turns."
)

print(
    "PASS: The notebook captured routing, tool, state, "
    "and grounding events."
)

MULTI-AGENT CONVERSATION SUMMARY
Persistent session: 59cdf4cb-a2aa-48d4-b8bd-847d0b3bab5d

Agents demonstrated:
 - root_agent
 - search_agent
 - weather_agent

Weather-turn transfers: ['weather_agent', 'search_agent']
Weather-turn tool calls: ['transfer_to_agent', 'get_weather_and_remember_location']
Recommendation-turn tool calls: []
Google Search grounding events: 1

PASS: Root, weather, and search agents were all demonstrated.
PASS: Conversation state persisted across turns.
PASS: The notebook captured routing, tool, state, and grounding events.


In [59]:
# Cell 18 — Interactive weather and local recommendation planner

print("=" * 72)
print("WEATHER AND LOCAL RECOMMENDATION PLANNER")
print("=" * 72)

await start_new_conversation()

user_city = input(
    "Enter a U.S. city: "
).strip()

user_state = input(
    "Enter a state name or abbreviation: "
).strip()

if not user_city or not user_state:
    print(
        "Both a city and state are required."
    )

else:
    location_request = (
        f"What is the current weather in "
        f"{user_city}, {user_state}? "
        "Include the next forecast period and "
        "any active weather alerts."
    )

    first_turn = (
        await send_conversation_message(
            location_request,
            show_events=True,
        )
    )

    user_interests = input(
        "\nYour response to search_agent: "
    ).strip()

    if not user_interests:
        print(
            "No activity preference was provided."
        )

    else:
        second_turn = (
            await send_conversation_message(
                (
                    f"I am interested in "
                    f"{user_interests}. "
                    "Use Google Search to recommend "
                    "current options near the location "
                    "saved earlier in this conversation."
                ),
                show_events=True,
            )
        )

        print("\n" + "=" * 72)
        print("PLANNING COMPLETE")
        print("=" * 72)

        involved_agents = sorted(
            set(
                first_turn["authors"]
                + second_turn["authors"]
            )
        )

        print(
            "Session:",
            conversation_session_id,
        )

        print(
            "Agents involved:",
            involved_agents,
        )

        while True:
            continue_choice = input(
                "\nWould you like to ask another "
                "question? (yes/no): "
            ).strip().lower()

            if continue_choice not in {
                "yes",
                "y",
            }:
                print("Conversation ended.")
                break

            next_message = input(
                "Enter your next question: "
            ).strip()

            if not next_message:
                print(
                    "No question was entered."
                )
                continue

            await send_conversation_message(
                next_message,
                show_events=True,
            )

WEATHER AND LOCAL RECOMMENDATION PLANNER
New conversation created: 765fe80e-5e7e-4884-852f-ef9b0052f103
Enter a U.S. city: Leesburg
Enter a state name or abbreviation: VA

USER
What is the current weather in Leesburg, VA? Include the next forecast period and any active weather alerts.

EVENT 1 | author=root_agent | branch=default | final=False
  TOOL CALL: transfer_to_agent
  ARGUMENTS: {'agent_name': 'weather_agent'}

EVENT 2 | author=root_agent | branch=default | final=False
  TRANSFER: weather_agent
  TOOL RESPONSE: transfer_to_agent

EVENT 3 | author=weather_agent | branch=default | final=False
  TOOL CALL: get_weather_and_remember_location
  ARGUMENTS: {'city': 'Leesburg', 'state': 'VA'}
[SESSION STATE] Saved location: Leesburg, VA

EVENT 4 | author=weather_agent | branch=default | final=False
  STATE UPDATE: {'selected_city': 'Leesburg', 'selected_state': 'VA', 'recommendation_stage': 'awaiting_user_interests'}
  TOOL RESPONSE: get_weather_and_remember_location

EVENT 5 | author=

The current temperature in Leesburg, VA is 71.6°F and conditions are clear with 37.9% humidity and winds from the west at 24.9 mph.

The forecast for this afternoon is sunny with a high near 70°F and a northwest wind around 6 mph.

There are no active weather alerts or advisories for Leesburg, VA.

What are you interested in doing in or near Leesburg, VA? For example, are you looking for restaurants, family activities, outdoor recreation, museums, shopping, nightlife, or local events?


Agents used: ['root_agent', 'weather_agent', 'search_agent']
Transfers: ['weather_agent', 'search_agent']
Tool calls: ['transfer_to_agent', 'get_weather_and_remember_location']
Grounded search events: 0

Your response to search_agent: Looking for restaurants

USER
I am interested in Looking for restaurants. Use Google Search to recommend current options near the location saved earlier in this conversation.

EVENT 1 | author=search_agent | branch=default | final=False
  TEXT: Here are some restaurant recommendations in or near Leesburg, VA:  *   **Tuscarora Mill** - This restaurant offers upscale American cuisine in a beautifully restored 19th-century grain mill. It's often recommended for special occasions or romantic meals and is affectionately referred to as "Tuskie's" by locals. It also has a wine cellar and log cabin available for events. *   **The Wine Kitchen on King** - A cozy bistro known for its extensive wine list and a seasonal menu featuring locally-sour
  GOOGLE SEARCH GR


FINAL AGENT RESPONSE


Here are some restaurant recommendations in or near Leesburg, VA:

*   **Tuscarora Mill** - This restaurant offers upscale American cuisine in a beautifully restored 19th-century grain mill. It's often recommended for special occasions or romantic meals and is affectionately referred to as "Tuskie's" by locals. It also has a wine cellar and log cabin available for events.
*   **The Wine Kitchen on King** - A cozy bistro known for its extensive wine list and a seasonal menu featuring locally-sourced ingredients. It's suggested for date nights and offers a curated tasting menu for a fancier option, with pasta made in-house.
*   **Firebirds Wood Fired Grill** - This American restaurant serves hand-cut steaks, fresh seafood, and more from a scratch kitchen. It offers an upscale dining experience at a modest price and is located in the Village at Leesburg near Cobb Village 12 Cinemas. They also have happy hour specials at their FIREBAR® and on the patio.
*   **Lightfoot Restaurant** - Housed in a grand old bank building across from the County Courthouse, Lightfoot is an elegant dinner spot serving refined American cuisine and an impressive wine list. It's recommended for special occasions or romantic meals, with a cozy spot by the window for lunch and happy hour specials at the bar.
*   **Twisted Cat Tavern** - This tavern serves American cuisine with a twist and delicious drinks in a cozy, homey atmosphere. It aims to reinvent classic dishes with its own flair.


Agents used: ['search_agent']
Transfers: []
Tool calls: []
Grounded search events: 1

PLANNING COMPLETE
Session: 765fe80e-5e7e-4884-852f-ef9b0052f103
Agents involved: ['root_agent', 'search_agent', 'weather_agent']

Would you like to ask another question? (yes/no): no
Conversation ended.
